# 2026-09-02_1 线1：Fig2 DEG 锚定 GSEA pathway（探索版，零阈值全记录）

目的：用 Fig2 的方案A DEG，锚定 GSEA pathway，缝合功能层与 DEG 层。

口径（本探索版）：
- 锚定 = pathway 在某 stage 的 **leading edge 基因** ∩ 同 celltype 同方向的 DM/DAL/AAM DEG；
- 方向由 NES 符号决定（NES>0=Macular，NES<0=Peripheral），leading edge 自带方向；
- **不设 NES 门槛、不设 padj 门槛**；是否显著仅作 `is_significant` 列记录，阈值全部后置；
- 双方向都跑，方向翻转打标，个案后续解读。

注意：DEG 与 GSEA rank 同源同一套 DESeq2 → 只能称 concordance/anchoring，非 validation。

In [ ]:
suppressPackageStartupMessages({library(dplyr); library(tidyr); library(readr); library(stringr)})

gsea_file <- "/project/work/2026-07-29_trajectory_class/Data/2026-08-02_1_lifecycle_directional_GSEA_all_results.csv"
deg_file  <- "/project/work/2026-09-01_DEG_summary/Data/2026-09-01_2_AUCell_signature_definition.csv"
out_dir   <- "/project/work/2026-09-01_GSEA_test/Data/"
stage_order <- c("Early","Mid","Late","Young","Old")

# 三命运短码：DEG 表用全称，脚本内统一为短码
sub_code <- c("Developmental maintained"="DM",
              "Developmental aging-lost"="DAL",
              "Adult acquired and maintained"="AAM")

gsea <- read_csv(gsea_file, show_col_types = FALSE)
deg  <- read_csv(deg_file,  show_col_types = FALSE)
cat("GSEA 行数:", nrow(gsea), "| DEG signature 行数:", nrow(deg), "\n")

In [ ]:
# ---- DEG 展开为长表：celltype × direction × subclass × gene ----------------
deg_long <- deg %>%
  filter(subclass %in% names(sub_code)) %>%
  transmute(celltype, direction = direction_class,
            subclass = unname(sub_code[subclass]), gene = signature_gene_names) %>%
  separate_rows(gene, sep = ";") %>%
  filter(gene != "") %>% distinct()

# ---- GSEA leading edge 展开为长表（仅 completed 行有 leadingEdge）----------
# 行方向由 NES 符号决定；is_significant 仅记录，不作过滤
gsea_le <- gsea %>%
  filter(!is.na(leadingEdge)) %>%
  mutate(direction = if_else(NES > 0, "Macular direction", "Peripheral direction"),
         is_significant = str_starts(gsea_status, "significant")) %>%
  select(celltype, database, pathway, stage, direction, NES, padj,
         gsea_status, is_significant, size, leadingEdge)

gsea_le_long <- gsea_le %>%
  separate_rows(leadingEdge, sep = ";") %>%
  rename(gene = leadingEdge) %>% filter(gene != "")

# ---- 求交：leading edge 基因 ∩ 同 celltype 同方向的命运 DEG ----------------
overlap <- gsea_le_long %>%
  inner_join(deg_long, by = c("celltype", "direction", "gene"))
cat("命中 (row×subclass×gene) 记录:", nrow(overlap), "\n")

In [ ]:
# ---- 输出1：锚定明细表（pathway × stage × subclass 一行）-------------------
detail <- overlap %>%
  group_by(celltype, database, pathway, stage, direction, subclass,
           NES, padj, gsea_status, is_significant, size) %>%
  summarise(n_anchor_genes = n(),
            anchor_genes = paste(sort(gene), collapse = ";"), .groups = "drop") %>%
  arrange(celltype, direction, pathway, subclass,
          factor(stage, levels = stage_order))
write_csv(detail, file.path(out_dir, "2026-09-02_1_anchor_detail_long.csv"), na = "NA")
cat("明细表行数:", nrow(detail), "\n")

In [ ]:
# 五期 NES / padj 宽表（供轨迹与显著性查看，含 Fig3 折线用）
nes_wide <- gsea %>% filter(stage %in% stage_order) %>%
  select(celltype, database, pathway, stage, NES) %>%
  pivot_wider(names_from = stage, values_from = NES, names_prefix = "NES_")
padj_wide <- gsea %>% filter(stage %in% stage_order) %>%
  select(celltype, database, pathway, stage, padj) %>%
  pivot_wider(names_from = stage, values_from = padj, names_prefix = "padj_")

# 方向翻转标记：同一 pathway 是否既有 Macular 显著又有 Peripheral 显著
flip <- gsea %>%
  group_by(celltype, database, pathway) %>%
  summarise(direction_flip = any(gsea_status=="significant_Central_macular") &
                             any(gsea_status=="significant_Peripheral"), .groups="drop")

In [ ]:
# ---- 输出2：pathway 汇总（celltype × pathway × direction × subclass）--------
uniq_genes <- function(x) sort(unique(unlist(strsplit(paste(x, collapse=";"), ";"))))
summary_tbl <- detail %>%
  group_by(celltype, database, pathway, direction, subclass, size) %>%
  summarise(n_stages_anchored    = n_distinct(stage),
            stages_anchored       = paste(sort(unique(stage)), collapse=";"),
            n_sig_stages_anchored = n_distinct(stage[is_significant]),
            anchored_at_sig_stage = any(is_significant),
            n_union_anchor_genes  = length(uniq_genes(anchor_genes)),
            union_anchor_genes    = paste(uniq_genes(anchor_genes), collapse=";"),
            .groups = "drop") %>%
  left_join(nes_wide,  by = c("celltype","database","pathway")) %>%
  left_join(padj_wide, by = c("celltype","database","pathway")) %>%
  left_join(flip,      by = c("celltype","database","pathway")) %>%
  arrange(celltype, direction, subclass, desc(anchored_at_sig_stage), desc(n_stages_anchored))
write_csv(summary_tbl, file.path(out_dir, "2026-09-02_1_anchor_pathway_summary.csv"), na = "NA")
cat("汇总表行数:", nrow(summary_tbl), "\n")

In [ ]:
# ---- 输出3：概览计数（anchored pathway 数：any stage / 至少一个显著 stage）--
counts <- summary_tbl %>%
  group_by(celltype, direction, subclass) %>%
  summarise(n_pathways_any_stage = n_distinct(pathway),
            n_pathways_sig_stage = n_distinct(pathway[anchored_at_sig_stage]),
            .groups = "drop") %>%
  arrange(direction, subclass, celltype)
write_csv(counts, file.path(out_dir, "2026-09-02_1_anchor_counts.csv"), na = "NA")

# ---- 输出4：多重归属矩阵（一条 pathway 被哪些命运同时锚定）------------------
multi <- summary_tbl %>%
  group_by(celltype, direction, pathway) %>%
  summarise(subclass_combo = paste(sort(unique(subclass)), collapse="+"),
            n_subclass = n_distinct(subclass), .groups="drop") %>%
  count(celltype, direction, subclass_combo, n_subclass, name = "n_pathways") %>%
  arrange(direction, celltype, desc(n_subclass))
write_csv(multi, file.path(out_dir, "2026-09-02_1_anchor_multiassign.csv"), na = "NA")

In [ ]:
# ---- 汇总打印（格式：any_stage 数，括号内为含显著 stage 数）-----------------
show_counts <- function(dir_label) {
  writeLines(paste0("=== anchored pathway 计数（", dir_label, "）==="))
  counts %>% filter(direction==dir_label) %>%
    mutate(v = paste0(n_pathways_any_stage," (",n_pathways_sig_stage,")")) %>%
    select(subclass, celltype, v) %>%
    pivot_wider(names_from=celltype, values_from=v, values_fill="0 (0)") %>% print()
}
show_counts("Macular direction")
show_counts("Peripheral direction")

writeLines("\n=== 多重归属组合（Macular 方向）===")
multi %>% filter(direction=="Macular direction") %>%
  group_by(subclass_combo) %>% summarise(n=sum(n_pathways), .groups="drop") %>%
  arrange(desc(n)) %>% print()

In [ ]:
# ---- 参数与 sessionInfo ----------------------------------------------------
writeLines(c("date_str: 2026-09-02_1  (线1 DEG 锚定 pathway 探索版)",
  paste0("gsea_file: ", gsea_file),
  paste0("deg_file: ", deg_file),
  "anchor: leading-edge gene INTERSECT 同 celltype 同方向 DM/DAL/AAM DEG (方案A refined)",
  "direction: 行方向由 NES 符号决定 (NES>0=Macular, NES<0=Peripheral)",
  "thresholds: 无 NES 门槛、无 padj 门槛（探索版全记录，is_significant 列标注）",
  "note: DEG 与 GSEA 同源 DESeq2 -> 只能称 concordance/anchoring, 非 validation",
  "", capture.output(sessionInfo())),
  file.path(out_dir, "2026-09-02_1_parameters_sessionInfo.txt"))
cat("完成。输出目录: /project/work/2026-09-01_GSEA_test/Data/\n")